# MiniTcl: análisis manual de scripts y análisis de expresiones con SLY
## Especificación y cuaderno de trabajo del estudiante

**Asignatura:** Compiladores · **Implementación:** Python 3.10 o posterior · **Enfoque:** orientación a objetos y `dataclasses`.

**Objetivo:** construir un lexer manual y un parser descendente recursivo para scripts MiniTcl, y un lexer con `sly.Lexer` y un parser LALR(1) con `sly.Parser` para las expresiones de `expr`. Ambos producen AST y no ejecutan el programa durante el análisis.

| Parte | Lexer | Parser |
|---|---|---|
| Scripts, comandos, palabras y cuerpos | Manual | `ScriptParser`, descendente recursivo |
| Expresiones de `expr` y condiciones de `if`, `while`, `for` | `ExprLexer(sly.Lexer)` | `ExprParser(sly.Parser)` |

Se entrega la especificación, el AST y esqueletos mínimos. **El estudiante completa las reglas a partir de la gramática.** El parser manual suministrado acepta solo palabras de un `TEXT`; el arranque de SLY acepta solo un número. No se entrega un intérprete ni un analizador completo. Las actividades se identifican con `TODO` o `NotImplementedError`; sus pruebas se activan por etapas.

El cuaderno se puede leer y ejecutar sin SLY: la demostración opcional de SLY se omite si no está instalado. Para desarrollar esa parte, instale `sly` en el entorno del kernel, por ejemplo con `%pip install sly` en una celda aparte, y reinicie el kernel si es necesario.

## 1. Alcance: un MiniTcl útil y realizable

La referencia pequeña es **Picol de antirez**, especialmente su enfoque didáctico; no se pretende comparar con todos los proyectos derivados que también se llaman Picol. Jim Tcl sirve como referencia de una implementación mucho más amplia. El tamaño de la gramática por sí solo no mide la capacidad de un Tcl: muchas ampliaciones son comandos incorporados, sin cambiar el parser de scripts.

Esta propuesta se llama **MiniTcl educativo 1.1**. Amplía el trabajo previo del curso con `for`, `foreach`, listas, diccionarios, operaciones de cadenas, arreglos de índice sencillo y una separación explícita entre AST estructural y validación de comandos.

| Obligatorio en el proyecto completo | Límite para mantener el proyecto realizable |
|---|---|
| Variables escalares y arreglos asociativos | Índices literales simples; sin sustituciones dentro del índice |
| Palabras sin agrupar, entre comillas y entre llaves | Sin expansión `{*}` ni concatenación después de cerrar una palabra agrupada |
| Interpolación y sustituciones `[...]` anidadas | Sin `eval`, `uplevel` ni generación dinámica de cuerpos de control |
| `expr`, condicionales, ciclos y procedimientos recursivos | Cuerpos y condiciones de control escritos entre llaves |
| Listas, diccionarios y operaciones sencillas de cadenas | Sin repertorio completo de Tcl ni bibliotecas de Jim Tcl |
| Ámbitos locales y declaración `global` | Sin namespaces, cierres, objetos ni trazas |
| `catch` y `error` | Sin opciones avanzadas de `return` ni de `catch` |

No se exigen GUI, sockets, procesos, archivos, extensiones nativas, bytecode ni generación para 6502 en esta entrega. El AST podrá servir después como entrada del checker, del intérprete y del generador de código.

**Tareas posibles:** factorial y Fibonacci recursivos; estadísticas de una lista; búsqueda lineal; histograma con diccionarios; tablas de multiplicar; procesamiento sencillo de texto y manejo de errores.

## 2. La decisión central: dos gramáticas y dos momentos

El parser estructural reconoce **scripts, comandos y palabras**. `if`, `set` y `proc` son nombres de comandos; no se convierten en palabras reservadas del lexer.

1. **Análisis estructural:** `Script → Command → Word → Part`. Las llaves producen una palabra literal; los corchetes contienen otro `Script`.
2. **Validación de comandos:** una pasada posterior comprueba aridad y las formas de los incorporados. En cuerpos y condiciones estáticos puede construir vistas `Script` y `Expr` adicionales.
3. **Ejecución futura:** resuelve variables, concatena partes de palabras y ejecuta comandos. Este paso no forma parte del trabajo del parser.

En `set mensaje {if algo}`, el bloque es texto; **no** es un `If`. En `while {$i < 10} {incr i}`, los argumentos son palabras literales que el comando interpreta como expresión y script, respectivamente.

Se recomienda conservar el AST estructural y asociarle un mapa de cuerpos y expresiones ya analizados. Un AST especializado con `If` o `While` puede añadirse después, pero no debe cambiar el significado de cualquier palabra entre llaves.

### Respecto de la especificación anterior del curso

Se conserva la precedencia de `expr`, incluido `-2 ** 2 = -(2 ** 2)`. Se precisan tres cambios: `#` abre comentario solo al comienzo de un comando; una palabra puede contener varias partes (`hola_$nombre`); y se conservan espacios significativos para distinguir palabras. No debe mezclarse este contrato con el lexer anterior sin adaptarlo.

### Frontera entre las dos herramientas

`ScriptParser` usa exclusivamente los tokens del lexer manual. El contenido literal de un argumento de `expr` se entrega a `parse_minitcl_expr()`, que usa `ExprLexer` y `ExprParser` de SLY. Las condiciones de `if`, `elseif`, `while` y `for` usan exactamente la misma fachada. Los cuerpos, la inicialización y el paso de `for` vuelven al parser manual.

En una expresión, el token `CMD_SUBST` guarda un `Script` construido por el parser manual al leer `[...]`. El parser SLY lo convierte en `SubstitutionExpr`; no ejecuta ese script. Por tanto, el cortocircuito sigue siendo posible.


## 3. Notación y reglas léxicas

En EBNF, `{ X }` significa repetición y `[ X ]` significa opcional. Los caracteres **entre comillas** son terminales: `"{"` representa una llave del programa, no una repetición de la notación. `ε` denota vacío.

### Contrato del lexer de scripts

El lexer debe conservar los límites de palabras. No basta con descartar todos los espacios.

| Token | Contenido / comportamiento |
|---|---|
| `TEXT` | Fragmento literal ya decodificado; incluye escapes procesados fuera de llaves |
| `VAR` | Nombre escalar; en un arreglo conserva también el índice literal |
| `BRACED` | Una palabra completa entre llaves; valor interior literal y span de toda la palabra |
| `QOPEN`, `QCLOSE` | Inicio y fin de palabra entre comillas |
| `LBRACKET`, `RBRACKET` | Abren y cierran una sustitución de comando |
| `WS` | Uno o más espacios horizontales fuera de palabras agrupadas |
| `SEP` | `;` o salto de línea fuera de llaves y comillas |
| `EOF` | Fin de la fuente, siempre presente |

`BRACED` se obtiene contando llaves anidadas; una expresión regular aislada no resuelve el balanceo general. Sus comillas y corchetes son texto. `\{` y `\}` no cambian la profundidad, y se conservan con su barra dentro de llaves. Dentro de comillas, espacios, `;` y saltos se incorporan a `TEXT`; `$` y `[` sí abren sustituciones. Una sustitución dentro de comillas vuelve temporalmente al modo script y luego restaura el modo comillas.

`{` y `"` agrupan solo al inicio de una palabra. Para simplificar, en una palabra no agrupada se rechazan llaves y comillas no escapadas; escribirlas mediante escapes. Tras cerrar una palabra agrupada solo puede seguir `WS`, `SEP`, `RBRACKET` o `EOF`.

`#` comienza un comentario solo donde podría empezar **un comando**, después de espacios iniciales. El comentario termina en el salto de línea; ese salto sigue separando comandos. `puts a#b` y `puts #dato` contienen texto, no comentarios. Al entrar en `[` empieza un nuevo contexto de comando.

Escapes obligatorios fuera de llaves: `\n`, `\t`, `\r`, `\\`, `\ `, `\$`, `\[`, `\]`, `\{`, `\}`, `\"`, `\;`, `\#`. Un escape desconocido es error en este perfil. Se normalizan CRLF/CR a LF antes del lexer; los offsets se refieren a esa fuente normalizada. **La continuación barra + salto de línea queda fuera del perfil obligatorio** y debe rechazarse explícitamente; puede agregarse como ampliación con mapa de posiciones.

### Variables y números

```ebnf
identifier = letter_or_underscore, { letter_or_underscore | digit } ;
variable   = "$", identifier, [ "(", index, ")" ]
           | "${", identifier, "}" ;
index      = index_char, { index_char } ;
```

Los identificadores usan letras ASCII. `index_char` es letra, dígito, `_`, `.`, `-` o `:`. Se aceptan `$a(3)` y `$a(nombre)`; se excluyen `$a($i)`, índices vacíos y arreglos multidimensionales. `${x}` desambigua `"${x}_fin"`. Un `$` sin nombre válido es error; para imprimirlo se escribe `\$`.

En scripts, `25` sigue siendo texto. Solo el lexer de expresiones genera `NUMBER`: `(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?`. El signo inicial es un operador; no pertenece al número. No hay hexadecimales ni números complejos en este perfil.

## 4. Gramática estructural de MiniTcl

Esta gramática opera sobre los tokens anteriores. El lexer no crea un `TEXT` vacío. `BRACED` puede tener contenido vacío y las comillas pueden delimitar una cadena vacía.

```ebnf
program       = script, EOF ;
script        = { WS | SEP }, [ command, { command_gap, command } ],
                { WS | SEP } ;
command_gap   = { WS }, SEP, { WS | SEP } ;
command       = word, { WS, word } ;
word          = BRACED | quoted_word | bare_word ;
quoted_word   = QOPEN, { part }, QCLOSE ;
bare_word     = part, { part } ;
part          = TEXT | VAR | substitution ;
substitution  = LBRACKET, script, RBRACKET ;
```

**Convención operativa para los espacios finales:** en `command`, después de consumir `WS`, solo se lee otra palabra si el siguiente token inicia `word`; si es un separador o el fin del contexto, termina el comando. El EBNF describe la secuencia válida; esta regla resuelve la decisión sobre el espacio sin retroceso.

`script` termina ante `EOF` en el nivel exterior o ante `RBRACKET` dentro de una sustitución. El método que abrió `[` consume el `RBRACKET`; el script interior lo deja pendiente. Un `RBRACKET` en el nivel exterior es error.

### ¿Por qué no usar `split()`?

```tcl
puts "Hola $nombre: [expr {2 + 3}]"
set dato {una palabra con espacios}
set ruta prefijo_${nombre}_fin
```

El primer comando tiene dos palabras, el segundo tres y el tercero tres. Los espacios dentro de agrupaciones no separan argumentos; las partes contiguas de una palabra se concatenan **al ejecutar**, sin crear nuevos argumentos.

### Recursión que sí es necesaria

`parse_substitution()` llama a `parse_script()`; este puede encontrar otra sustitución. Eso es descenso recursivo válido. No es recursividad izquierda: antes de la llamada se consumió `LBRACKET`.

## 5. Gramática de expresiones para SLY, con AST

Esta parte se implementa con **`sly.Lexer` y `sly.Parser`**, que construye un analizador LALR(1). Se utiliza una gramática por niveles, con recursividad izquierda para los operadores asociativos a la izquierda. No debe traducirse a métodos descendentes recursivos.

```ebnf
expression     = logical_or ;
logical_or     = logical_or, "||", logical_and | logical_and ;
logical_and    = logical_and, "&&", equality | equality ;
equality       = equality, ("==" | "!=" | "eq" | "ne"), relational | relational ;
relational     = relational, ("<" | "<=" | ">" | ">="), additive | additive ;
additive       = additive, ("+" | "-"), multiplicative | multiplicative ;
multiplicative = multiplicative, ("*" | "/" | "%"), unary | unary ;
unary          = ("+" | "-" | "!"), unary | power ;
power          = primary, "**", unary | primary ;
primary        = NUMBER | BOOL | VAR | STRING
               | "(", expression, ")" | CMD_SUBST ;
```

SLY recibe el fin de entrada al agotarse el iterador de tokens: **no se crea un token `EOF` para SLY**. Su símbolo inicial será `expression`; debe rechazar residuos como `2 3`. El lexer manual y su cursor sí conservan el `EOF` definido para scripts.

Los niveles de esta gramática ya definen precedencia y asociatividad. No hace falta una tabla `precedence` adicional. Si el estudiante opta por una gramática plana con tabla de precedencia, deberá justificarla y conservar exactamente los AST de las pruebas, especialmente la relación entre unarios y potencia.

| Grafía / categoría | Nombre de token recomendado para SLY |
|---|---|
| `||`, `&&` | `OR`, `AND` |
| `==`, `!=`, `eq`, `ne` | `EQ`, `NE`, `STREQ`, `STRNE` |
| `<`, `<=`, `>`, `>=` | `LT`, `LE`, `GT`, `GE` |
| `+`, `-`, `*`, `/`, `%`, `**`, `!` | `PLUS`, `MINUS`, `TIMES`, `DIVIDE`, `MOD`, `POWER`, `NOT` |
| `(`, `)` | `LPAREN`, `RPAREN` |
| Número, booleano, variable y cadena literal | `NUMBER`, `BOOL`, `VAR`, `STRING` |
| Sustitución de comando balanceada | `CMD_SUBST`, cuyo `value` será un `Script` |

El lexer de SLY debe definir los patrones largos antes de los cortos: `**` antes de `*`, `<=` antes de `<`, etc. `eq`, `ne` y los booleanos se reconocen como palabras completas. Los blancos fuera de cadenas se ignoran en el lexer de expresiones, conservando los datos necesarios para calcular posiciones. No se incorporan comentarios de script dentro de `expr`.

`BOOL` acepta `true`, `false`, `yes`, `no`, `on`, `off`; `0` y `1` son números convertibles a booleano. `STRING` es una cadena entre comillas literal, sin interpolación en este perfil; rechazar `$` o `[` no escapados dentro de ella. No hay cadenas desnudas, funciones matemáticas, ternario ni operaciones de bits en la entrega obligatoria.

### Contrato para `CMD_SUBST`

Al encontrar `[`, una acción del lexer SLY delega la lectura balanceada al analizador manual mediante un callback. Este devuelve el AST del script y el índice inmediatamente posterior al `]`. La acción devuelve un único token `CMD_SUBST` y actualiza `self.index` para continuar después del cierre. No se busca simplemente el siguiente `]` ni se usa una expresión regular para balancear todo el script. Debe respetarse el anidamiento y los modos de llaves/comillas del lexer manual.

Firma sugerida del callback:

```python
read_substitution(text, opening_index, base_offset, source_name) -> tuple[Script, int]
```

`opening_index` y el índice devuelto son locales a `text`; los spans del AST se construyen sumando `base_offset`. Al avanzar sobre una sustitución multilínea, el lexer SLY también actualiza su contador de líneas. Cada llamada anidada a la fachada de expresiones crea sus propias instancias de lexer/parser, para no sobrescribir un análisis en curso.

| Entrada | Forma del AST esperada |
|---|---|
| `2 + 3 * 4` | `Binary('+', 2, Binary('*', 3, 4))` |
| `10 - 3 - 2` | `Binary('-', Binary('-', 10, 3), 2)` |
| `2 ** 3 ** 2` | `Binary('**', 2, Binary('**', 3, 2))` |
| `-2 ** 2` | `Unary('-', Binary('**', 2, 2))` |
| `2 ** -3` | `Binary('**', 2, Unary('-', 3))` |
| `$x != 0 && 10 / $x > 1` | `Binary('&&', comparación, comparación)` |

Las comparaciones encadenadas se agrupan por la izquierda. Las acciones de SLY construyen nodos, no calculan resultados. El evaluador posterior hace el cortocircuito; no usar `eval()` de Python.

## 6. Contrato de comandos: formas que debe validar la siguiente pasada

En esta tabla, `word` representa **una sola palabra** del comando, `body` es una palabra `BRACED` reinterpretada como script y `test` es una palabra `BRACED` reinterpretada como expresión. El estudiante implementa el parser genérico primero; estas formas no son alternativas del lexer.

```ebnf
set_cmd      = "set", name, [ word ] ;
unset_cmd    = "unset", name ;
incr_cmd     = "incr", name, [ word ] ;
append_cmd   = "append", name, word, { word } ;
puts_cmd     = "puts", [ "-nonewline" ], word ;
expr_cmd     = "expr", test ;
if_cmd       = "if", test, [ "then" ], body,
               { "elseif", test, [ "then" ], body }, [ "else", body ] ;
while_cmd    = "while", test, body ;
for_cmd      = "for", body, test, body, body ;
foreach_cmd  = "foreach", identifier, word, body ;
proc_cmd     = "proc", identifier, parameters, body ;
parameters   = BRACED ;   (* su contenido: lista plana de identificadores únicos *)
return_cmd   = "return", [ word ] ;
break_cmd    = "break" ;
continue_cmd = "continue" ;
global_cmd   = "global", identifier, { identifier } ;
catch_cmd    = "catch", body, [ identifier ] ;
error_cmd    = "error", word ;
call_cmd     = word, { word } ;
```

`call_cmd` es la forma estructural general y no una alternativa que deba probarse después de fallar con un incorporado. `name` es un nombre escalar o una referencia escrita `a(indice)` con el mismo índice simple del léxico. En `set a(3) 8` no se escribe `$` en el nombre que recibe el comando.

**Perfil estático del curso:** nombres de incorporados no redefinibles; nombres de procedimientos y variables receptoras de asignación escritos literalmente; condiciones y cuerpos entre llaves; `expr` recibe exactamente una expresión entre llaves. Los nombres de comandos calculados son válidos estructuralmente pero el validador los rechaza como fuera del perfil. `set accion puts; $accion hola` ilustra esa diferencia. Los argumentos de una llamada sí pueden usar interpolación y sustituciones.

Los parámetros no tienen valores por defecto ni `args` variádico. `foreach` tiene exactamente una variable y una lista; no se exigen formas de múltiples listas. `global` se permite dentro de procedimientos y enlaza un nombre con el entorno global.

### Biblioteca mínima de datos

| Forma | Semántica que deberá tener el intérprete futuro |
|---|---|
| `list ?valor ...?` | Construye una representación de lista que preserve cada elemento |
| `llength lista` | Número de elementos |
| `lindex lista indice` | Índice entero base 0; fuera de rango devuelve cadena vacía |
| `lappend nombre ?valor ...?` | Añade elementos; crea lista vacía si no existe la variable |
| `lset nombre indice valor` | Reemplaza un elemento existente; índice fuera de rango es error |
| `dict create ?clave valor ...?` | Diccionario serializado como pares; claves repetidas conservan el último valor |
| `dict get diccionario clave` | Devuelve valor; clave ausente es error |
| `dict exists diccionario clave` | Devuelve `0` o `1` |
| `dict set nombre clave valor` | Modifica variable diccionario; la crea si no existe |
| `string length texto` | Número de caracteres Unicode |
| `string tolower texto` / `string toupper texto` | Conversión de mayúsculas/minúsculas |
| `string compare a b` | Devuelve `-1`, `0` o `1`, orden lexicográfico por puntos de código |
| `split texto ?separadores?` | Separa por cualquiera de los caracteres dados; por defecto espacio, tabulador y salto |
| `join lista ?separador?` | Une elementos; separador por defecto un espacio |

En `split`, separadores contiguos producen elementos vacíos; un separador vacío divide en caracteres. En `dict create` se exige un número par de argumentos. Los arreglos `a(k)` son variables asociativas; los diccionarios son valores. No se necesita un comando `array` para la entrega obligatoria.

Estas operaciones amplían las tareas posibles sin ampliar la gramática estructural del lenguaje.

## 7. Las listas requieren su propio lector

El contenido de una lista no es un script: `$x`, `[comando]` y `;` son datos, sin sustituciones ni separación de comandos. El salto de línea separa elementos, igual que un espacio.

```ebnf
list_value    = { LIST_WS }, [ element, { LIST_WS, element } ], { LIST_WS } ;
element       = LIST_BRACED | LIST_QUOTED | LIST_BARE ;
```

`LIST_WS` contiene uno o más blancos. El lector de listas balancea llaves, elimina agrupaciones y procesa los escapes admitidos fuera de llaves; no evalúa variables ni comandos. Tras una agrupación exige blanco o fin. En elementos no agrupados se rechazan llaves y comillas no escapadas, como simplificación coherente con el script.

Contrato esencial: `parse_list(format_list(values)) == values`. No construir listas mediante `' '.join(values)` cuando los elementos contienen espacios, llaves o cadenas vacías. El lector de parámetros de `proc` puede reutilizar el lector de listas y después exigir identificadores únicos. No se entrega su implementación.

## 8. Modelo del AST

Se entrega el modelo de datos para que la actividad se concentre en traducir la gramática. `Span` usa offsets semiabiertos `[start, end)` en la fuente normalizada e incluye el nombre de la fuente. En `BRACED`, el span de `Literal` cubre el interior y el de `Word` incluye las llaves.

| Nodo | Responsabilidad |
|---|---|
| `Script` | Secuencia de comandos |
| `Command` | Secuencia de palabras; la primera es el nombre del comando |
| `Word` | Conserva modo `bare`, `quoted` o `braced` y sus partes |
| `Literal` | Texto que no debe resolverse como variable o comando |
| `Variable` | Referencia escalar o de arreglo con índice literal |
| `CommandSubstitution` | Script cuyo resultado será parte de una palabra |
| `Constant`, `VariableExpr`, `SubstitutionExpr` | Hojas del AST de expresiones |
| `Unary`, `Binary` | Operaciones sin evaluar |

Una palabra entre llaves tiene exactamente un `Literal`, incluso si es vacío. Una palabra entre comillas vacía puede tener cero partes. Los espacios de separación y los paréntesis de agrupación no necesitan nodos propios.

In [1]:
from __future__ import annotations
from dataclasses import dataclass, fields, is_dataclass
from typing import Literal as Mode

@dataclass(frozen=True)
class Span:
    start: int
    end: int
    source: str = "<input>"

@dataclass(frozen=True, kw_only=True)
class Node:
    span: Span

@dataclass(frozen=True)
class Literal(Node):
    text: str

@dataclass(frozen=True)
class Variable(Node):
    name: str
    index: str | None = None

@dataclass(frozen=True)
class CommandSubstitution(Node):
    script: Script

Part = Literal | Variable | CommandSubstitution

@dataclass(frozen=True)
class Word(Node):
    parts: tuple[Part, ...]
    mode: Mode["bare", "quoted", "braced"] = "bare"

@dataclass(frozen=True)
class Command(Node):
    words: tuple[Word, ...]

@dataclass(frozen=True)
class Script(Node):
    commands: tuple[Command, ...]

@dataclass(frozen=True, kw_only=True)
class Expr(Node):
    pass

@dataclass(frozen=True)
class Constant(Expr):
    value: int | float | str | bool

@dataclass(frozen=True)
class VariableExpr(Expr):
    variable: Variable

@dataclass(frozen=True)
class SubstitutionExpr(Expr):
    script: Script

@dataclass(frozen=True)
class Unary(Expr):
    operator: str
    operand: Expr

@dataclass(frozen=True)
class Binary(Expr):
    operator: str
    left: Expr
    right: Expr

### Un AST construido a mano para comprender el resultado

La siguiente celda **no analiza texto**. Muestra el AST que se espera obtener para `puts hola`. Observe que no se imprime `hola` como efecto del programa: únicamente se inspecciona su estructura.

In [2]:
def show_ast(value, indent=0):
    """Impresión estructurada sin Graphviz ni dependencias externas."""
    prefix = " " * indent
    if is_dataclass(value):
        print(prefix + type(value).__name__)
        for field in fields(value):
            if field.name == "span":
                continue
            item = getattr(value, field.name)
            print(prefix + "  " + field.name + ":")
            show_ast(item, indent + 4)
    elif isinstance(value, tuple):
        for item in value:
            show_ast(item, indent)
    else:
        print(prefix + repr(value))

example_ast = Script(
    commands=(Command(words=(
        Word(parts=(Literal(text="puts", span=Span(0, 4)),), span=Span(0, 4)),
        Word(parts=(Literal(text="hola", span=Span(5, 9)),), span=Span(5, 9)),
    ), span=Span(0, 9)),), span=Span(0, 9))
show_ast(example_ast)

Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'puts'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'hola'
          mode:
            'bare'


## 9. Infraestructura mínima: tokens, cursor y diagnósticos

El lexer deberá producir la secuencia de `Token`. No se entrega un lexer completo. Para `VAR`, `value` contiene el nombre e `index` el índice opcional; para `BRACED`, `value` es el contenido interior sin decodificar. La implementación posterior debe conservar la fuente original normalizada para reanalizar los cuerpos con offsets correctos.

El cursor permite mirar, aceptar o exigir tokens. Los métodos del estudiante deben producir `ParseError` para entradas incorrectas; `NotImplementedError` significa una actividad todavía pendiente, no un error del programa MiniTcl.

In [3]:
@dataclass(frozen=True)
class Token:
    kind: str
    value: str
    span: Span
    index: str | None = None

class ParseError(Exception):
    def __init__(self, message: str, span: Span):
        self.message = message
        self.span = span
        super().__init__(f"{span.source}:{span.start}: {message}")

class TokenCursor:
    def __init__(self, tokens):
        self.tokens = tuple(tokens)
        if not self.tokens or self.tokens[-1].kind != "EOF":
            raise ValueError("El lexer debe agregar EOF")
        self.position = 0

    @property
    def current(self):
        return self.tokens[self.position]

    def accept(self, kind):
        if self.current.kind != kind:
            return None
        token = self.current
        if kind != "EOF":
            self.position += 1
        return token

    def expect(self, kind):
        token = self.accept(kind)
        if token is None:
            raise ParseError(
                f"Se esperaba {kind}; se encontró {self.current.kind}",
                self.current.span)
        return token

    def skip(self, *kinds):
        while self.current.kind in kinds and self.current.kind != "EOF":
            self.position += 1


def cover(first: Span, last: Span):
    if first.source != last.source:
        raise ValueError("No combinar posiciones de fuentes diferentes")
    return Span(first.start, last.end, first.source)

## 10. Esqueleto del parser: completar desde la gramática

Solo se resuelve el caso de una palabra que consta de **un `TEXT`**. Se deja implementada la coordinación básica de scripts y comandos para mostrar el uso del cursor. El estudiante debe ampliar `parse_word`, desarrollar los métodos pendientes y escribir el lexer.

El argumento `stop` fija el fin del contexto, pero no consume ese token. En particular, el analizador exterior exige `EOF` y el de sustitución deberá exigir `RBRACKET`. Cada iteración de un bucle tiene que consumir al menos un token o terminar: un error habitual es quedar atrapado en un bucle infinito.

In [4]:
class ScriptParser(TokenCursor):
    WORD_START = {"TEXT", "VAR", "BRACED", "QOPEN", "LBRACKET"}

    def parse(self) -> Script:
        result = self.parse_script(stop="EOF")
        self.expect("EOF")
        return result

    def parse_script(self, stop="EOF") -> Script:
        start = self.current.span
        commands = []
        self.skip("WS", "SEP")
        while self.current.kind != stop:
            if self.current.kind == "EOF":
                raise ParseError(f"Falta {stop}", self.current.span)
            commands.append(self.parse_command())
            if self.current.kind == stop:
                break
            self.expect("SEP")
            self.skip("WS", "SEP")
        end = commands[-1].span if commands else self.current.span
        return Script(commands=tuple(commands), span=cover(start, end))

    def parse_command(self) -> Command:
        words = [self.parse_word()]
        while self.accept("WS"):
            if self.current.kind not in self.WORD_START:
                break
            words.append(self.parse_word())
        return Command(words=tuple(words), span=cover(words[0].span, words[-1].span))

    def parse_word(self) -> Word:
        if self.current.kind == "BRACED":
            return self.parse_braced_word()
        if self.current.kind == "QOPEN":
            return self.parse_quoted_word()
        if self.current.kind in {"TEXT", "VAR", "LBRACKET"}:
            return self.parse_bare_word()
        raise ParseError(
            f"Se esperaba una palabra; se encontró {self.current.kind}",
            self.current.span)

    def parse_bare_word(self) -> Word:
        parts = [self.parse_part()]
        while self.current.kind in {"TEXT", "VAR", "LBRACKET"}:
            parts.append(self.parse_part())
        return Word(
            parts=tuple(parts),
            mode="bare",
            span=cover(parts[0].span, parts[-1].span))

    def parse_quoted_word(self) -> Word:
        opening = self.expect("QOPEN")
        parts = []
        while self.current.kind in {"TEXT", "VAR", "LBRACKET"}:
            parts.append(self.parse_part())
        closing = self.expect("QCLOSE")
        return Word(
            parts=tuple(parts),
            mode="quoted",
            span=cover(opening.span, closing.span))

    def parse_braced_word(self) -> Word:
        token = self.expect("BRACED")
        part = Literal(text=token.value, span=token.span)
        return Word(parts=(part,), mode="braced", span=token.span)

    def parse_part(self) -> Part:
        if self.current.kind == "TEXT":
            token = self.expect("TEXT")
            return Literal(text=token.value, span=token.span)
        if self.current.kind == "VAR":
            token = self.expect("VAR")
            return Variable(name=token.value, index=token.index, span=token.span)
        if self.current.kind == "LBRACKET":
            return self.parse_substitution()
        raise ParseError(
            f"Se esperaba una parte de palabra; se encontró {self.current.kind}",
            self.current.span)

    def parse_substitution(self) -> CommandSubstitution:
        opening = self.expect("LBRACKET")
        script = self.parse_script(stop="RBRACKET")
        closing = self.expect("RBRACKET")
        return CommandSubstitution(
            script=script,
            span=cover(opening.span, closing.span))

In [5]:
# Tokens MANUALES: esta celda no sustituye el trabajo del lexer.
# Fuente equivalente: set x 10; puts hola
starter_tokens = [
    Token("TEXT", "set", Span(0, 3)), Token("WS", " ", Span(3, 4)),
    Token("TEXT", "x", Span(4, 5)), Token("WS", " ", Span(5, 6)),
    Token("TEXT", "10", Span(6, 8)), Token("SEP", ";", Span(8, 9)),
    Token("WS", " ", Span(9, 10)), Token("TEXT", "puts", Span(10, 14)),
    Token("WS", " ", Span(14, 15)), Token("TEXT", "hola", Span(15, 19)),
    Token("EOF", "", Span(19, 19)),
]
starter_ast = ScriptParser(starter_tokens).parse()
assert len(starter_ast.commands) == 2
assert starter_ast.commands[0].words[2].parts[0].text == "10"
show_ast(starter_ast)

Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'set'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'x'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                '10'
          mode:
            'bare'
    Command
      words:
        Word
          parts:
            Literal
              text:
                'puts'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'hola'
          mode:
            'bare'


## 11. Contratos de los métodos pendientes

| Método | Entrada / decisión | Salida y obligación |
|---|---|---|
| `parse_word` | Mirar primer token | Elegir `BRACED`, `QOPEN` o palabra no agrupada |
| `parse_bare_word` | `TEXT`, `VAR` o `LBRACKET` | Acumular partes hasta que deje de comenzar una parte; no consumir separador |
| `parse_quoted_word` | `QOPEN` | Admitir cero partes; exigir `QCLOSE`; rechazar texto adyacente tras el cierre |
| `parse_braced_word` | `BRACED` | Un `Word(mode='braced')` con un solo `Literal`; no interpretar su contenido |
| `parse_part` | Primera clase de fragmento | Producir `Literal`, `Variable` o `CommandSubstitution` |
| `parse_substitution` | `LBRACKET` | Llamar a `parse_script(stop='RBRACKET')`, consumir cierre y construir nodo |
| `parse_command` | Inicio de palabra | Acumular palabras separadas por `WS`; no confundir partes contiguas con argumentos |

### Puntos de predicción que debe justificar

- `FIRST(word) = {BRACED, QOPEN, TEXT, VAR, LBRACKET}`.
- `FIRST(part) = {TEXT, VAR, LBRACKET}`.
- Después de `QOPEN`, `QCLOSE` permite una palabra vacía.
- Después de `WS` en un comando, mirar si empieza otra palabra; de lo contrario el espacio es final.
- Después de `LBRACKET`, un `RBRACKET` inmediato admite un script vacío.

**Actividad escrita:** explique por qué el modo del lexer forma parte del contrato. No declare que toda la sintaxis de Tcl es una gramática LL(1) sobre caracteres sin mostrar cómo se tokenizan los contextos.

## 12. Esqueleto mínimo de expresiones con SLY

Se entrega únicamente el caso de una expresión formada por un número. `ExprLexer` hereda de `sly.Lexer` y `ExprParser` de `sly.Parser`; no heredan de `TokenCursor`.

**Actividad:** incorporar el repertorio de tokens de la sección 5, definir todas las producciones con `@_()` y devolver los nodos del AST. El lexer de scripts y su parser descendente recursivo siguen siendo manuales. No reutilizar los tokens del cursor manual como si fueran tokens SLY: tienen contratos distintos.

La celda es opcional cuando SLY no está instalado. No se simula la biblioteca con clases sustitutas. El arranque solo acepta un número, por lo que las pruebas completas siguen desactivadas hasta terminar las reglas.

In [6]:
%pip install sly

Note: you may need to restart the kernel to use updated packages.


In [7]:
try:
    from sly import Lexer, Parser
except ImportError:
    SLY_AVAILABLE = False
    print("SLY no está instalado: se omite el arranque. Para la actividad use %pip install sly.")
else:
    SLY_AVAILABLE = True

    class ExprLexer(Lexer):
        # ARRANQUE: solo NUMBER; ampliar tokens y reglas según sección 5.
        tokens = {NUMBER, CMD_SUBST}
        ignore = ' \t\r'
        NUMBER = r'(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?'

        def __init__(self, read_substitution=None, base_offset=0, source_name='<expr>'):
            self.base_offset = base_offset
            self.source_name = source_name
            self.read_substitution = read_substitution

        @_(r'\[')
        def CMD_SUBST(self, token):
            if self.read_substitution is None:
                position = self.base_offset + token.index
                raise ParseError(
                    "CMD_SUBST requiere el callback read_substitution",
                    Span(position, position + 1, self.source_name))
            script, next_index = self.read_substitution(
                self.text,
                token.index,
                self.base_offset,
                self.source_name)
            token.value = script
            self.index = next_index
            return token

        def error(self, token):
            position = self.base_offset + token.index
            raise ParseError("Carácter inválido o token aún no implementado en expr",
                             Span(position, position + 1, self.source_name))

    class ExprParser(Parser):
        tokens = ExprLexer.tokens
        start = 'expression'

        def __init__(self, text='', base_offset=0, source_name='<expr>'):
            self.text = text
            self.base_offset = base_offset
            self.source_name = source_name

        @_('NUMBER')
        def expression(self, p):
            raw = p.NUMBER
            value = float(raw) if any(c in raw for c in '.eE') else int(raw)
            start = self.base_offset + p.index
            return Constant(value=value,
                            span=Span(start, start + len(raw), self.source_name))

        # TODO: sustituir la única producción por la gramática completa
        # de la sección 5; las acciones construyen Unary, Binary, etc.

        def error(self, token):
            if token is None:
                position = self.base_offset + len(self.text)
                raise ParseError("Fin inesperado de expresión",
                                 Span(position, position, self.source_name))
            position = self.base_offset + token.index
            raise ParseError(f"Token inesperado en expr: {token.type}",
                             Span(position, position + 1, self.source_name))

    number_lexer = ExprLexer()
    number_parser = ExprParser(text='42')
    number_ast = number_parser.parse(number_lexer.tokenize('42'))
    assert isinstance(number_ast, Constant) and number_ast.value == 42
    print('Arranque SLY: NUMBER -> Constant(42). Reglas restantes pendientes.')

Arranque SLY: NUMBER -> Constant(42). Reglas restantes pendientes.


### Cómo traducir las producciones a reglas de SLY

Traduzca cada alternativa de la sección 5 a una producción decorada con `@_()`. En una regla binaria como `additive PLUS multiplicative`, la acción debe construir `Binary` con los dos AST y un span que cubra la operación. En una regla de paso, como `additive -> multiplicative`, devuelva el AST ya construido.

**Actividad:** explicar cómo la recursividad izquierda de `additive` produce la asociación de `10 - 3 - 2`, y cómo `power -> primary POWER unary` produce la asociación de `2 ** 3 ** 2`. Esta recursividad izquierda es válida en SLY; no debe introducirse en el parser descendente recursivo de scripts.

No se entrega una acción por cada operador: el estudiante debe completarlas. Dibuje antes los AST de `8 / 2 * 3`, `2 ** 3 ** 2`, `-(4 + 1)` y `$a(3) + [doble 2]`, sin evaluarlos.

## 13. Análisis de cuerpos y condiciones, sin ejecutarlos

El AST estructural basta para guardar el programa, pero una entrega completa del análisis sintáctico debe demostrar cómo reutilizar ambos parsers en los contextos estáticos.

| Comando | Argumentos que se vuelven a analizar |
|---|---|
| `expr` | Expresión única |
| `if` / `elseif` | Cada condición con `ExprParser` de SLY; cada cuerpo con `ScriptParser` manual |
| `while` | Condición como expresión; cuerpo como script |
| `for` | Inicio, paso y cuerpo como scripts; condición como expresión |
| `foreach` | Cuerpo como script; lista sigue siendo una palabra ordinaria |
| `proc` | Parámetros como lista de nombres; cuerpo como script |
| `catch` | Cuerpo como script |

**Actividad:** escribir `analyze_command(command)` y una estructura de resultados que vincule cada argumento con su AST adicional. Mantener el AST estructural original. En la pasada estática se pueden analizar cuerpos no ejecutados; un error de sintaxis dentro de un cuerpo estático debe reportarse aunque no se tome esa rama.

Al reanalizar un `BRACED`, la primera posición interior es `word.span.start + 1`; propague ese offset y el nombre de la fuente. No desplace las posiciones calculándolas sobre texto interpolado: el perfil exige cuerpos literales. No interprete como scripts los bloques de `set`, `puts`, `list` o los datos de `dict`.

Los corchetes ya producen un `Script` durante el análisis estructural. Visite también esos scripts al validar. Una llamada a un procedimiento definido más adelante no es por sí sola un error sintáctico.

### Fachada común para expresiones

`parse_minitcl_expr(source, *, base_offset=0, source_name='<expr>') -> Expr` integra `ExprLexer` y `ExprParser` de SLY y consume toda la entrada. No recibe `expr` ni las llaves exteriores, solo su contenido literal. El analizador de comandos la llama para `expr`, `if`, `elseif`, `while` y `for`; no debe existir un segundo parser manual para condiciones.

En `set r [expr {$x + 3}]`, el lexer manual conserva el argumento de `expr` como `BRACED`. La pasada de análisis entrega `$x + 3` a SLY y asocia su `Binary` al argumento original. En `expr {$x + [doble 2]}`, el lexer SLY devuelve `VAR`, `PLUS`, `CMD_SUBST`; ese último token contiene el `Script` producido por el analizador manual. Ningún paso consulta el valor de `x` ni ejecuta `doble`.


## 14. Semántica mínima para que el AST tenga un significado claro

Estas reglas son el contrato para la etapa posterior; **no se pide implementar el evaluador en este cuaderno**.

- Los valores visibles son cadenas; números, listas y diccionarios se convierten al usarlos. El intérprete puede conservar representaciones internas auxiliares.
- Las palabras ordinarias evalúan sus partes de izquierda a derecha y concatenan sus resultados. El resultado de una sustitución no se vuelve a dividir en argumentos ni se vuelve a interpretar como código.
- Una palabra `braced` entrega su literal. Una condición entre llaves se analiza una vez como expresión y se evalúa cada vez que el control la necesita; las variables no se congelan al entrar al ciclo.
- Cada llamada de procedimiento crea un ámbito local; `global` enlaza nombres globales. Un arreglo tiene la misma política de ámbito que una variable escalar. No se permite que un nombre sea simultáneamente escalar y arreglo.
- Un procedimiento devuelve el valor explícito de `return` o el del último comando. Un script vacío devuelve `""`. `set`, `incr`, `append`, `lappend`, `lset` y `dict set` devuelven el nuevo valor. `unset`, `puts`, `global` y `proc` devuelven `""`.
- `if` devuelve el resultado del cuerpo elegido o `""` si no se elige ninguno. `while`, `for` y `foreach` devuelven `""`.
- `return` solo es válido en procedimientos. `break` y `continue` requieren un ciclo en el procedimiento actual; no pueden escapar hacia un ciclo del llamador. En un `for`, `continue` en el cuerpo ejecuta el paso antes de la siguiente condición. En `foreach`, pasa al siguiente elemento.
- `catch {script} ?variable?` devuelve `0` para terminación normal y `1` para error; guarda resultado o mensaje si se da variable. En este perfil, `return`, `break` y `continue` se propagan, no se convierten a códigos adicionales. Esto es una simplificación respecto de Tcl.
- `error mensaje` interrumpe con un error. Variable inexistente, conversión imposible o aridad inválida son errores semánticos, no fallos de tokenización.

### Operadores

`+ - * **` operan con números; `/` devuelve real aun con enteros; `%` requiere enteros y usa el resto `a - floor(a/b)*b`. División o módulo por cero producen error. Se excluyen resultados complejos y no finitos. `< <= > >=` comparan números. `== !=` comparan numéricamente si ambos operandos admiten conversión; en otro caso comparan cadenas. `eq ne` comparan siempre cadenas.

La conversión booleana admite números finitos (cero falso, otro valor verdadero) y las palabras booleanas en minúsculas del contrato léxico. `&&`, `||` y `!` devuelven `0` o `1`. No se evalúa el hijo derecho de `&&` si el izquierdo es falso, ni el de `||` si el izquierdo es verdadero, **incluidas sus sustituciones de comandos**. `**` tiene la precedencia definida en la sección 5, aunque otras implementaciones de Tcl difieran.

## 15. Programas de aceptación: tareas interesantes

Los siguientes programas son **fuentes de prueba**, no código Python ejecutable. En esta etapa deben producir AST; los resultados indicados serán comprobables cuando exista el intérprete.

### A. Recursión y sustituciones anidadas

```tcl
proc factorial {n} {
    if {$n <= 1} {return 1}
    return [expr {$n * [factorial [expr {$n - 1}]]}]
}
puts [factorial 6]
```

Resultado futuro: `720` (se admite representación numérica equivalente). El cuerpo de `proc` es literal hasta la pasada de análisis de comandos. La expresión del retorno contiene un script que, a su vez, contiene otra expresión.

### B. Estadísticas de una lista

```tcl
set datos [list 4 8 15 16 23 42]
set suma 0
foreach dato $datos {
    set suma [expr {$suma + $dato}]
}
set promedio [expr {$suma / [llength $datos]}]
puts "Promedio: $promedio"
```

Resultado futuro: promedio `18.0`. `datos` es una lista almacenada como valor; `foreach` no analiza su contenido como código.

### C. Histograma con diccionarios

```tcl
set conteo [dict create]
foreach palabra [split {sol luna sol mar luna sol}] {
    if {[dict exists $conteo $palabra]} {
        dict set conteo $palabra [expr {[dict get $conteo $palabra] + 1}]
    } else {
        dict set conteo $palabra 1
    }
}
puts [dict get $conteo sol]
```

Resultado futuro: `3`. Las claves son valores; el comando `dict set` recibe literalmente el nombre de la variable que modifica.

### D. Ciclo `for`, arreglo y errores

```tcl
set tabla(3) 9
for {set i 0} {$i < 5} {incr i} {
    if {$i == 2} {continue}
    puts [expr {$i * $i}]
}
set codigo [catch {expr {10 / 0}} mensaje]
puts "$codigo: $mensaje"
puts $tabla(3)
```

Resultado futuro: cuadrados `0`, `1`, `9`, `16`; código de error `1` y un diagnóstico de división por cero; finalmente `9`.

**Actividad:** elija uno, enumere todas sus invocaciones recursivas de análisis y localice las palabras cuyos contenidos deben analizarse de nuevo.

## 16. Pruebas por etapas

No basta con aceptar entradas válidas: compare la **forma** del AST, la cantidad de palabras y el contenido de los literales. Las pruebas del esqueleto se ejecutan ahora. El resto se activa cuando el estudiante implemente las dos fachadas que se indican.

In [8]:
# Pruebas del material suministrado: no requieren completar actividades.
assert ScriptParser([Token("EOF", "", Span(0, 0))]).parse().commands == ()
trailing = [Token("TEXT", "puts", Span(0, 4)),
            Token("WS", " ", Span(4, 5)), Token("EOF", "", Span(5, 5))]
assert len(ScriptParser(trailing).parse().commands[0].words) == 1
try:
    ScriptParser([Token("RBRACKET", "]", Span(0, 1)),
                  Token("EOF", "", Span(1, 1))]).parse()
except (ParseError, NotImplementedError):
    # El esqueleto aún no incorpora todos los diagnósticos.
    pass
else:
    raise AssertionError("No se debe aceptar un cierre exterior")
print("Comprobaciones básicas del esqueleto: correctas.")

Comprobaciones básicas del esqueleto: correctas.


In [9]:
# Fachadas que desarrollará el estudiante. Mantener estas firmas.
def parse_minitcl(source: str) -> Script:
    raise NotImplementedError("Integrar lexer manual de scripts y ScriptParser descendente recursivo")

def parse_minitcl_expr(source: str, *, base_offset=0, source_name='<expr>') -> Expr:
    raise NotImplementedError("Integrar ExprLexer(sly.Lexer) y ExprParser(sly.Parser), con la gramática completa")

def expr_shape(node):
    """Proyección para comparar estructura sin depender de offsets."""
    if isinstance(node, Constant):
        return node.value
    if isinstance(node, Unary):
        return (node.operator, expr_shape(node.operand))
    if isinstance(node, Binary):
        return (node.operator, expr_shape(node.left), expr_shape(node.right))
    raise TypeError(type(node).__name__)

RUN_STUDENT_TESTS = False  # Activar cuando estén listas las dos fachadas.
if RUN_STUDENT_TESTS:
    script = parse_minitcl('puts "Hola $nombre"; set x {a b}')
    assert len(script.commands) == 2
    assert len(script.commands[0].words) == 2
    greeting = script.commands[0].words[1]
    assert greeting.mode == "quoted"
    assert [type(p).__name__ for p in greeting.parts] == ["Literal", "Variable"]
    block = script.commands[1].words[2]
    assert block.mode == "braced" and block.parts[0].text == "a b"
    assert expr_shape(parse_minitcl_expr("2 + 3 * 4")) == ("+", 2, ("*", 3, 4))
    assert expr_shape(parse_minitcl_expr("10 - 3 - 2")) == ("-", ("-", 10, 3), 2)
    assert expr_shape(parse_minitcl_expr("2 ** 3 ** 2")) == ("**", 2, ("**", 3, 2))
    assert expr_shape(parse_minitcl_expr("-2 ** 2")) == ("-", ("**", 2, 2))
    assert expr_shape(parse_minitcl_expr("2 ** -3")) == ("**", 2, ("-", 3))
    print("Pruebas del estudiante: correctas.")
else:
    print("Pruebas completas pendientes: habilitar RUN_STUDENT_TESTS al terminar.")

Pruebas completas pendientes: habilitar RUN_STUDENT_TESTS al terminar.


### Matriz adicional que debe convertir en pruebas

| Entrada | Resultado esperado / nivel |
|---|---|
| `puts ""` y `puts {}` | Dos palabras; argumento vacío, modos diferentes |
| `set x hola_${nombre}_fin` | Tres palabras; la última tiene literal, variable y literal |
| `puts [set x 1; set x]` | Sustitución con dos comandos |
| `puts [list [set x] y]` | Sustituciones anidadas |
| `puts {a {b} c}` | Literal `a {b} c`, sin comandos internos |
| `puts {a\{b\}c}` | Literal conserva las barras y no altera balanceo |
| `puts a#b` / `puts #dato` | `#` dentro del argumento, sin comentario |
| `# comentario` | Script vacío |
| `puts "a; b"` | `;` dentro del literal, sin separar comandos |
| `puts $a(3)` | Referencia al arreglo `a`, índice literal `3` |
| `puts $a($i)` | Fuera del perfil léxico; error localizado |
| `puts {sin cierre` | Error de llave faltante, señalar apertura |
| `puts "sin cierre` | Error de comillas faltantes |
| `puts [set x 1` | Error de corchete faltante |
| `puts {a}b` / `puts "a"b` | Error por texto después de una palabra agrupada |
| `puts ]` | Error: cierre fuera de contexto |
| `set x 1 2` | AST estructural válido; error de aridad en validación |
| `if {1} {puts si} else` | AST estructural válido; falta cuerpo en validación |
| `expr {2 + }` | Script válido; error al analizar su expresión |
| `expr {2 3}` | Error de expresión por residuos |
| `expr {0 && [error inesperado]}` | AST válido; futuro evaluador no ejecuta `error` |
| `proc f {x x} {return $x}` | Error de parámetros duplicados en validación |
| `puts {if mal formado}` | Válido; ese argumento es texto, no un condicional |

Para errores de entrada, las fachadas públicas deben lanzar `ParseError` (incluidos errores del lexer, o una subclase). No capture cualquier `Exception` en las pruebas negativas: ocultaría errores de programación o actividades aún sin resolver.

## 17. Diagnósticos y estrategia de implementación

Cada diagnóstico debe contener nombre de fuente, línea, columna, elemento esperado y elemento encontrado. El cursor suministrado conserva offsets; **actividad:** calcular línea y columna a partir de la fuente normalizada, y mostrar el fragmento afectado. En un cierre faltante conviene incluir también la posición de apertura.

Orden de trabajo sugerido:

1. Lexer de texto, espacios, separadores y comentarios; comprobar el esqueleto con tokens reales.
2. `BRACED`, variables y palabras con múltiples partes.
3. Comillas y corchetes anidados; probar cambios de contexto y restauración del modo.
4. `ExprLexer(sly.Lexer)` y `ExprParser(sly.Parser)`: tokens, producciones LALR(1), acciones de AST y errores.
5. Validador de formas de comandos y reanálisis de cuerpos y condiciones.
6. Lector de listas, parámetros de procedimientos y casos de integración.
7. Posiciones completas, mensajes y pruebas negativas.

Deténgase en el primer error para esta entrega. La recuperación por sincronización es opcional: no descarte tokens hasta el primer `;` sin considerar comillas y anidamiento.

### Preguntas que deben acompañar la entrega

1. ¿Por qué `set x {$y}` no debe generar una referencia a `y`?
2. ¿Qué evita que `hola_$nombre` se convierta en dos argumentos?
3. ¿Qué parte del sistema distingue `set n 12` de `expr {12}`?
4. ¿Por qué la gramática de SLY admite recursividad izquierda para la suma, mientras la potencia asocia a la derecha?
5. ¿Cómo conserva el análisis de `0 && [error fallo]` la posibilidad de cortocircuito?
6. ¿Qué hace el lexer con `]` dentro de llaves y con `;` dentro de comillas?
7. ¿Qué errores puede detectar el parser genérico y cuáles requieren conocer el comando?
8. ¿Cómo cambian los contextos de `break` y `return` al entrar a un nuevo procedimiento?

## 18. Entregables y criterios de evaluación

Entregar el cuaderno completado, los módulos equivalentes (`model.py`, `script_lexer.py`, `script_parser.py`, `expr_lexer.py`, `expr_parser.py`, `command_analyzer.py`) y pruebas reproducibles. Los nombres de código se escriben en inglés y las explicaciones en español.

| Criterio | Peso |
|---|---:|
| Gramática aplicada y explicación de decisiones predictivas | 15 % |
| Léxico contextual, balanceo y límites de palabras | 20 % |
| Parser estructural y construcción correcta del AST | 25 % |
| Lexer y parser con SLY: AST, precedencia y asociatividad | 20 % |
| Validación de comandos y análisis de cuerpos estáticos | 10 % |
| Diagnósticos y pruebas positivas/negativas | 10 % |

**Condición de terminación:** no quedan `NotImplementedError` en las rutas obligatorias, se puede analizar cada programa de la sección 15 sin ejecutarlo, se obtienen AST verificables y se rechazan los casos inválidos en la etapa apropiada. El intérprete y el generador de código serán trabajos posteriores.

La entrega debe demostrar que `ExprLexer` deriva de `sly.Lexer` y `ExprParser` de `sly.Parser`, que la gramática completa genera tablas sin conflictos no justificados y que sus acciones producen AST. Debe comprobarse que tanto `expr` como las condiciones pasan por la misma fachada SLY. Los scripts se analizan con el lexer manual y descenso recursivo.


## 19. Referencias y decisiones de diseño

Fuentes primarias consultadas para delimitar el lenguaje:

- [Reglas de sintaxis de Tcl](https://www.tcl-lang.org/man/tcl8.7/TclCmd/Tcl.html): separación entre comandos, palabras, agrupación y sustituciones.
- [Picol de antirez](https://github.com/antirez/picol): referencia de intérprete pequeño y didáctico. Existen versiones y derivados con capacidades distintas; no se asume que todos tengan el mismo repertorio.
- [Jim Tcl](https://jim.tcl-lang.org/home/doc/www/www/index.html) y [su documentación](https://jim.tcl-lang.org/home/doc/trunk/jim_tcl.txt): referencia de una implementación con un alcance considerablemente mayor.

La gramática y los contratos de este cuaderno son una **propuesta docente propia**, no una transcripción del manual de Tcl ni una promesa de compatibilidad completa. Los límites de índices, escapes, cuerpos estáticos, `expr`, listas y `catch` son decisiones de este perfil. El tamaño intermedio se busca por el alcance de la biblioteca y la arquitectura; no por una cifra de líneas de código.

Este cuaderno retoma el alcance y la precedencia descritos en la especificación MiniTcl 1.0 del curso y documenta explícitamente sus ampliaciones. Las fuentes externas sirven para contextualizar; el contrato obligatorio para el estudiante es el definido aquí.

- [Documentación oficial de SLY](https://sly.readthedocs.io/en/latest/sly.html): clases `Lexer`/`Parser`, producciones, posiciones y generación LALR(1).


## 20. EBNF para diagramas railroad: dos entradas y un puente

El archivo `MiniTcl_Railroad.ebnf` usa notación de estilo W3C para el generador de Bottlecaps. `manual_script_input` inicia la gramática del lexer manual y el parser descendente recursivo. `sly_expression_input` inicia la gramática LALR(1) de SLY; su fin de entrada es implícito.

Los diagramas de `builtin_forms` son contratos de validación, no alternativas nuevas del parser estructural. `expr_argument`, `test` y `body` aparecen como `BRACED`: el análisis de comandos entrega el contenido de los dos primeros a SLY y el de `body` al parser manual. Es una delegación entre analizadores, no la inserción de la gramática de expresiones dentro del lexer de scripts.

`CMD_SUBST` representa un token SLY cuyo valor es un AST `Script`; el patrón de caracteres `[...]` se muestra en `substitution`, dentro de la gramática manual. Las reglas auxiliares `variable_syntax`, `number_syntax` y `boolean_syntax` explican la forma del texto; no son producciones adicionales de SLY ni sustituyen sus expresiones regulares. Los nombres de tokens entre comillas son etiquetas para el diagrama.

Las producciones de expresiones conservan recursividad izquierda intencionadamente: son para SLY. Las de scripts conservan repeticiones para facilitar el descenso recursivo.

Copie el siguiente bloque en https://www.bottlecaps.de/rr/ui:

```ebnf
manual_script_input ::= program
program ::= script 'EOF'

script ::= ('WS' | 'SEP')* (command (command_gap command)*)? ('WS' | 'SEP')*
command_gap ::= 'WS'* 'SEP' ('WS' | 'SEP')*
command ::= word ('WS' word)*
word ::= 'BRACED' | quoted_word | bare_word
quoted_word ::= 'QOPEN' part* 'QCLOSE'
bare_word ::= part+
part ::= 'TEXT' | 'VAR' | substitution
substitution ::= '[' script ']'

sly_expression_input ::= expression
expression ::= logical_or
logical_or ::= logical_or '||' logical_and | logical_and
logical_and ::= logical_and '&&' equality | equality
equality ::= equality ('==' | '!=' | 'eq' | 'ne') relational | relational
relational ::= relational ('<' | '<=' | '>' | '>=') additive | additive
additive ::= additive ('+' | '-') multiplicative | multiplicative
multiplicative ::= multiplicative ('*' | '/' | '%') unary | unary
unary ::= ('+' | '-' | '!') unary | power
power ::= primary '**' unary | primary
primary ::= 'NUMBER' | 'BOOL' | 'VAR' | 'STRING' | '(' expression ')' | 'CMD_SUBST'

builtin_forms ::= set_cmd | unset_cmd | incr_cmd | append_cmd | puts_cmd
                | expr_cmd | if_cmd | while_cmd | for_cmd | foreach_cmd
                | proc_cmd | return_cmd | break_cmd | continue_cmd
                | global_cmd | catch_cmd | error_cmd
                | list_cmd | llength_cmd | lindex_cmd | lappend_cmd | lset_cmd
                | dict_cmd | string_cmd | split_cmd | join_cmd

set_cmd ::= 'set' 'WS' name ('WS' word)?
unset_cmd ::= 'unset' 'WS' name
incr_cmd ::= 'incr' 'WS' name ('WS' word)?
append_cmd ::= 'append' 'WS' name ('WS' word)+
puts_cmd ::= 'puts' 'WS' ('-nonewline' 'WS')? word
expr_cmd ::= 'expr' 'WS' expr_argument
if_cmd ::= 'if' 'WS' test ('WS' 'then')? 'WS' body
           ('WS' 'elseif' 'WS' test ('WS' 'then')? 'WS' body)*
           ('WS' 'else' 'WS' body)?
while_cmd ::= 'while' 'WS' test 'WS' body
for_cmd ::= 'for' 'WS' body 'WS' test 'WS' body 'WS' body
foreach_cmd ::= 'foreach' 'WS' identifier 'WS' word 'WS' body
proc_cmd ::= 'proc' 'WS' identifier 'WS' parameters 'WS' body
return_cmd ::= 'return' ('WS' word)?
break_cmd ::= 'break'
continue_cmd ::= 'continue'
global_cmd ::= 'global' ('WS' identifier)+
catch_cmd ::= 'catch' 'WS' body ('WS' identifier)?
error_cmd ::= 'error' 'WS' word
call_cmd ::= word ('WS' word)*

body ::= 'BRACED'
expr_argument ::= 'BRACED'
test ::= expr_argument
parameters ::= 'BRACED'
parameter_content ::= 'LIST_WS'* (identifier ('LIST_WS' identifier)*)? 'LIST_WS'*
name ::= identifier ('(' index ')')?

list_cmd ::= 'list' ('WS' word)*
llength_cmd ::= 'llength' 'WS' word
lindex_cmd ::= 'lindex' 'WS' word 'WS' word
lappend_cmd ::= 'lappend' 'WS' name ('WS' word)*
lset_cmd ::= 'lset' 'WS' name 'WS' word 'WS' word
dict_cmd ::= dict_create_cmd | dict_get_cmd | dict_exists_cmd | dict_set_cmd
dict_create_cmd ::= 'dict' 'WS' 'create' ('WS' word 'WS' word)*
dict_get_cmd ::= 'dict' 'WS' 'get' 'WS' word 'WS' word
dict_exists_cmd ::= 'dict' 'WS' 'exists' 'WS' word 'WS' word
dict_set_cmd ::= 'dict' 'WS' 'set' 'WS' name 'WS' word 'WS' word
string_cmd ::= string_length_cmd | string_case_cmd | string_compare_cmd
string_length_cmd ::= 'string' 'WS' 'length' 'WS' word
string_case_cmd ::= 'string' 'WS' ('tolower' | 'toupper') 'WS' word
string_compare_cmd ::= 'string' 'WS' 'compare' 'WS' word 'WS' word
split_cmd ::= 'split' 'WS' word ('WS' word)?
join_cmd ::= 'join' 'WS' word ('WS' word)?

list_value ::= 'LIST_WS'* (element ('LIST_WS' element)*)? 'LIST_WS'*
element ::= 'LIST_BRACED' | 'LIST_QUOTED' | 'LIST_BARE'

variable_syntax ::= '$' identifier ('(' index ')')? | '${' identifier '}'
identifier ::= letter_or_underscore (letter_or_underscore | digit)*
letter_or_underscore ::= [A-Z] | [a-z] | '_'
digit ::= [0-9]
index ::= index_char+
index_char ::= letter_or_underscore | digit | '.' | '-' | ':'
number_syntax ::= (digit+ ('.' digit*)? | '.' digit+) exponent?
exponent ::= ('e' | 'E') ('+' | '-')? digit+
boolean_syntax ::= 'true' | 'false' | 'yes' | 'no' | 'on' | 'off'
```